# Simulating MUSE NFM field
This example walks through the complete workflow to use TipTorch model for post-processing a MUSE Narrow Field Mode (NFM) observation block (OB). `MUSEObservation` is the main interface for simulating OB which can:

- read OB data
- remove data artifacts (e.g., bad pixels)
- simulate PSFs
- simulate the entire field with multipe (point) sources
- disentangle source spectra and separate beackround (constant)
- fit the PSF
- save the results.


## Overview
The general workflow is:

1. Initialize the OB from the raw and reduced cubes, then save cache.
2. Initialize the sources.
3. Initialize the simulation by predicting the initial PSFs, extracting source information, and computing the bad-pixel mask.
4. Simulate the field using either sparse or full spectra.
5. Optionally fit the PSF model and simulate the field again.
6. Save the results for later reuse.
7. Analyze the results.

The steps after 3 can be run in a different order when needed.

## Configuration

By default, TipTorch stores all configs, calibrations, and settings in `~/.tiptorch`. There, the `project_config.json` can be found. Edit `~/.tiptorch/project_config.json` to set the default device computation device (CPU or GPU) and other project settings, like global directories definitions. See `README.md` for more info.

In [ ]:
%reload_ext autoreload
%autoreload 2
%run ../tools/notebook_setup.py

from tiptorch._config import default_device as device, project_settings
from tools.observations import MUSEObservation
from pathlib import Path

 ## Inputs

The `MUSEObservation` requires:
 - a raw MUSE NFM data cube
 - a reduced MUSE NFM data cube

The two cubes should be temporally associated. If the reduced cube combines several OBs, provide any relevant raw file from the observation block. The initial PSF estimate may then be less accurate in this case, but it can later be improved through PSF fitting if the field contains point sources.
The raw headers and reduced cube are used to generate a cached  file containing the required OB metadata. Once created, this cache is reused by subsequent runs.

In [ ]:
# Define the location of the NFM data. It can live anywhere. One convenient option is to add the path to the project configuration file
MUSE_DATA_FOLDER = Path(project_settings["MUSE_data_folder"])

data_folder = MUSE_DATA_FOLDER / 'clumpy_galaxies/'

# Set the paths to the raw, reduced, and cached MUSE NFM cubes below.
cube_path  = data_folder / "reduced_cubes/CUBE_0002.fits"
raw_path   = data_folder / "raw_data/MUSE.2023-04-27T05_11_59.783.fits.fz"
cache_path = data_folder / "cached_cubes/CUBE_0002.pickle"

ob = MUSEObservation(raw_path, cube_path, cache_path, device=device)

## Defining sources

Adding sources just tells e model where in the field so simualte PSFs. You can detect sources automatically, supply them manually with `AddSources()`, or add by modifying the `sources_table` (see below). TipTorch's built-in source detection is intentionally basic. Thus, for more robust results, source detection can be delegated to a specialized tool. Sources should be added before calling `InitSimulation()`. Otherwise, simulation will be automatically re-initialized.

>**Note:** currently, `AddSources()`, requires input coordinates to be defined in pixels. This will change soon.

To fit the PSF in one area while simulating sources elsewhere, add those sources with `AddSource(..., weight=0.0)`. A zero weight excludes a source from fitting while retaining it in the simulation. This is useful when a field contains a bright point source but the science target lies elsewhere. The model extrapolates the PSF shape to the target position automatically.

In [ ]:
ob.DetectSources(nsigma=35, threshold='auto')

# Add a fake source. The first argument is a list of [x, y] pixel coordinates, the second is a list of weights for each source.
ob.AddSources([[100, 200]], weights=[0.0])
ob.DisplayField()

In [ ]:
# Delete non-existent source
ob.DeleteSources([1])  # Example: delete sources with IDs 1 and 2
ob.DisplayField()

### Supplying a custom source table (WIP)

Assign the resulting table to `ob.sources_table` before calling `InitSimulation()` to skip the built-in detection step or to manually modify the list of sources.

The table must contain at least these columns:

- `ID`: source identifier; it does not need to be numeric or literal (i.e. object label)
- `x_peak`, `y_peak`: source coordinates in pixels (WCS support is planned)
- `peak_value`: approximate mean source flux, used to distinguish bright and faint sources for initialization; fluxes are re-estimated during simulation
- `weight`: relative contribution to PSF fitting; the default is `1.0`
- `src_id`: created automatically, a technical field that maps to source index in arrays


## Simulation initialization

The `InitSimulation()` function:
- initializes the PSF model
- predicts the PSF shape from OB metadata and system telemetry
- extracts source spectra and metadata at the table coordinates
- computes the bad-pixel mask

Call it after detecting sources and making any changes to the source table. Changes to source data made later require `InitSimulation()` to be run again.

This function also computes spectra for each source by averaging within $5\times5$ pixels window around each PSF core. At this stage, spectra are only approximate because crowded fields may require flux disentangling. Full disentangling is performed later in this example with `SimulateField()`.

In [ ]:
ob.InitSimulation()

Predicting a PSF directly from OB metadata makes it possible to process fields with no point sources. However, if point sources are present, fitting can further improve the estimate.


## PSF fitting

Refine the initial PSF estimate using point sources in the science field, options:
- _PSF_ - fits parameters defining the PSF's shape
- _astrometry_ - fine-tunes sub-pixel shifts of sources (always recommended)
- _photometry_ - refines flux-normalizing coefficients for each source

In [ ]:
ob.FitPSFModel(fit=['PSF', 'astrometry', 'photometry'], repeat=3, max_iter=200)

Repeating the fit a couple of times can help to "escape" local minima. This example uses maximalist settings since in many cases, `repeat=2` with `max_iter=75–100` is sufficient.

Fitting can also be limited to astrometry and/or photometry only:

```python
ob.FitPSFModel(fit=['astrometry'], repeat=3, max_iter=200)
ob.FitPSFModel(fit=['astrometry', 'photometry'], repeat=3, max_iter=200)
```

## Sparse-spectrum simulation

`SimulateField()` is the main simulation function:

- Renders the scientific field with all defined sources
- Disentangles flux from overlapping source
- (Optional) generates the individual PSF for every source position.

Sources spectra are disentangled by solving a linear system:
$$
\sum_\lambda \mathrm{PSF}_\lambda \, F_\lambda + b_\lambda = I_{\mathrm{data}, \, \lambda}
$$
with respect to $F_\lambda$. This is particularly helpful in crowded fields, although it is not critical for this example. In any case, running this function is crucial to compute _accurate_ true spectra.

In [ ]:
model = ob.SimulateField(full_spectrum=False) # Sparse simulation by default. Set full_spectrum=True to simulate the full spectrum of each source.

Note that the simulations support both *sparse* and *full* regimes. Expensive operations such as PSF fitting use sparse data to reduce runtime and resource consumption. The sparse representation contains 7 spectrally averaged bins with $\Delta\lambda = \mathrm{13.75-14}$ nm, the same spatial resolution is the same.
`full_spectrum=False` selects this sparse representation and is the default. Use the full spectrum for the final scientific analysis.
Sparse cubes are also used for quick previews. Display the sparse-spectrum simulation. The `plot_profiles` option adds the median source profiles and their fits / preditions.

In [ ]:
ob.DisplaySimulation(plot_profiles=True, full_spectrum=False)

## Full-spectrum simulation

The model parameters estimated from sparse spectra can now be interpolated to the full-spectrum simulation.


In [ ]:
_ = ob.SimulateField(full_spectrum=True, disentangle_spectra=True)
ob.DisplaySimulation(plot_profiles=True, full_spectrum=True, focus_on_src=0)  # Focus view on the first source with id=0
ob.PlotSourceSpectra()

## Save and restore the simulation

Save the fitted state without the full cubes, then demonstrate how to load it into a fresh `MUSEObservation` instance and rerun the full-spectrum simulation.

In [ ]:
_ = ob.SaveState(data_folder/'metadata'/(cache_path.stem+'.pkl'), save_full_cubes=False)

Test re-loaded OB:

In [ ]:
ob = MUSEObservation(raw_path, cube_path, cache_path, device=device)

ob.LoadState(data_folder/'metadata'/(cache_path.stem+'.pkl'), device=device)
_ = ob.SimulateField(full_spectrum=True)
ob.DisplaySimulation(plot_profiles=True, full_spectrum=True, focus_on_src=0)  # Focus on the first source